In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
with highway_stats as (

    select  traffic.segment_id,
            aadt_total,
            aadt_ev,
            aadt_truck,
            peak_factor,
            segment.interstate,
            segment.lanes,
            segment.speed_limit,
            
            risk_score,
            
            crash_rate, 
            incident_rate, 
    
            (traffic.aadt_ev / traffic.aadt_total) * 100 as ev_percentage,
            (traffic.aadt_truck / traffic.aadt_total) * 100 as truck_percentage
    
    from    data5035.spring26.traffic_counts as traffic
    
            join data5035.spring26.road_segments as segment
                on segment.segment_id = traffic.segment_id
            
            left join data5035.spring26.weather_risk as weather
                on weather.segment_id = traffic.segment_id 
        
            left outer join data5035.spring26.incidents as i
                on i.segment_id = segment.segment_id 

),wetland_areas as ( 
    select  r.segment_id,
            count(w.constraint_id) as wetlands_nearby
    from    data5035.spring26.road_segments r
            left join data5035.spring26.env_constraints w
                on ST_DWITHIN(r.geom, w.geom, 1000) -- 1 km environmental buffer.
    group   by r.segment_id
),
distances as ( 
    select  s.segment_id,
            i.interchange_id, 
            st_distance(
                s.geom,
                i.geom
            ) / 1000 as distance_km
    from    data5035.spring26.road_segments s
            cross join data5035.spring26.interchanges i   
), interchanges_ranked as (
    select  segment_id, interchange_id, 
            rank() over (partition by interchange_id order by distance_km) as ranked 
    from    distances 
), closested_segment_to_interchange as (
    select  d.segment_id, d.interchange_id, d.distance_km
    from    interchanges_ranked ir
            join distances d 
                on d.segment_id = ir.segment_id and 
                   d.interchange_id = ir.interchange_id 
    where   ranked = 1                
), interchanges as ( 
    select  segment_id, count(distinct interchange_id) as interchange_rates
    from    closested_segment_to_interchange
    group   by segment_id
) 


select  hwy.segment_id,
        aadt_total,
        aadt_ev,
        aadt_truck,
        peak_factor,
        interstate,
        lanes,
        speed_limit,
    
        /* high ev usage (more evs = better) */
        ntile(10) over (
            order by ev_percentage desc
        ) as high_ev_vehicle_usage,
    
        /* low truck usage (less trucks = better) */
        ntile(10) over (
            order by truck_percentage asc
        ) as low_truck_usage,
    
        /* peak factor */
        ntile(10) over (
            order by peak_factor desc
        ) as peak_time_bucket,
    
        /* weather risk (lower risk = better) */
        ntile(10) over (
            order by risk_score asc
        ) as weather_risk_area,
    
        wetlands_nearby,
    
        ntile(10) over (
            order by crash_rate asc
        ) as crash_rate,
    
        ntile(10) over (
            order by incident_rate asc
        ) as incident_rate,
    
        ntile(10) over (
            order by interchange_rates asc 
        ) as interchange_rates

from    highway_stats hwy 

        left outer join wetland_areas wet 
            on wet.segment_id = hwy.segment_id 
            
        left outer join interchanges i 
            on i.segment_id = hwy.segment_id 

""") 


df = df.to_pandas()
 
df = df[df["WETLANDS_NEARBY"] == 0]

df = df[df["HIGH_EV_VEHICLE_USAGE"] >= 4]
df = df[df["INCIDENT_RATE"] < 8]
df = df[df["CRASH_RATE"] < 8]
df = df[df["WEATHER_RISK_AREA"] < 8] 
 
df["score"] = (
    df["CRASH_RATE"] +
    df["INCIDENT_RATE"] +
    df["WEATHER_RISK_AREA"] +
    df["PEAK_TIME_BUCKET"] + 
    df["INTERCHANGE_RATES"] - 
    df["LOW_TRUCK_USAGE"] -
    df["HIGH_EV_VEHICLE_USAGE"]
)
 
df_sorted = df.sort_values("score")
 
best_sites = df_sorted.head(4)

print(best_sites.to_csv(index=False))
 

  